# SleePyLand scoring for several files

> This file is intended to be run inside the sleepyland architecture.  
> To run, clone the [repo](https://github.com/biomedical-signal-processing/sleepyland), [download and run](https://hub.docker.com/r/bspsupsi/sleepyland) sleepyland, and insert this file into the my-notebook folder.  
> Put the original EDFs in a folder `raw_edf` and the manually scored EDF(s) in a folder `manual_scored`, **both next to this notebook** (i.e. inside `my-notebook/`, not in `../input`: the sleepyland GUI empties `/app/input` on every run).  
> Manual scorings are renamed once, at the start of section 7, from `xx.edf` to `xx_scored.edf`; a scoring is matched to the recording with the same name `xx.edf` (or, as a fallback, the one with the longest common name prefix).

What this file does:  
1. set configurations
2. preprocess data (de-saturate samples --> band-pass filter --> resample to 128Hz --> clip outliers --> export); preprocessing statistics are recorded per file
3. prepare for sleepyland
4. load manual scoring
5. define YASA, statistics (hypnogram & metrics) and plot functions
6. define scoring function
7. score all EDFs of the folder (one after the other) with all available models
8. (optional) look at results of every single file: plot, stats and compare with manual scoring
9. report across files: averages of all metrics from section 8 and of the preprocessing statistics; saved to `report_across_files.xlsx`

## 1. set configurations

In [11]:
#%pip install openpyxl
#%pip install edfio

import os
os.environ["MNE_USE_NUMBA"] = "false"   #set before mne is imported

import re, glob, shutil, traceback, mne
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import requests, edfio
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# -------------------- CONFIGURE ME --------------------
# --- Folders (sleepyland's predict_one reads only from API_DIR = /app/input/myedf)
RAW_DIR    = Path("raw_edf")            # ORIGINAL EDF files: folder next to this notebook (my-notebook/raw_edf), not searched recursively
API_DIR    = Path("../input/myedf")     # = /app/input/myedf in the container. Must contain only the file being scored.
MANUAL_DIR = Path("manual_scored")      # optional: manually scored EDF(+) files, folder next to this notebook (annotations are enough)
AUTO_MOVE_ORIGINALS = True              # EDFs found in API_DIR are moved to RAW_DIR (never deleted)
RENAME_MANUAL = True                    # rename every manual scoring in MANUAL_DIR from xx.edf to xx_scored.edf (once, at the start of section 7)

EPOCH_SEC     = 30                      # epoch length in seconds
EXPORT_SET    = False                   # also export EEGLAB .set (needs: pip install eeglabio)
FORCE_RESCORE = True                    # True = ignore the "already scored" skip rule (skipped files are NOT part of the report in section 9)
SHOW_PER_FILE = False                   # section 8: True = show tables and plots for every single file (long), False = only the report in section 9

# Channels, first match wins (case-insensitive).
EEG_CANDIDATES = ["FpZ", "EEG"]
EOG_CANDIDATES = ["EOG1", "EOG2", "EOG"]   # maybe do ["EOG2", "EOG1"] as a second run to check quality
EMG_CANDIDATES = ["EMG1", "EMG2", "EMG"]   # only used for the optional native YASA run

# --- Signal preparation for sleepyland
BANDPASS        = (0.3, 35.0)    # Hz, zero-phase FIR. None = off (NOT recommended for this device: huge DC drift)
RESAMPLE_HZ     = 128            # sleepyland resamples to 128 Hz internally anyway
DESATURATE      = True           # linearly interpolate samples at the ADC limit before filtering (avoids filter ringing)
SATURATION_UV   = 180_000.0      # device range is +-187.5 mV
CLIP_IQR_FACTOR = 20.0           # clip at +-20*IQR after filtering (sleepyland uses the same rule)
EEG_LABEL       = "C4"           # renaming the channel: Container-YASA skips non-central EEG (FpZ!). Deep models only look at the type "EEG".
EOG_LABEL       = "EOG"
KEEP_PREPARED   = True           # keep the prepared EDF in OUT_DIR/prepared for inspection

# --- Scoring
MODELS        = ["usleep", "deepresnet", "transformer", "yasa", "ensemble"]
PRIMARY_MODEL = "ensemble"          # its hypnogram is written into the scored EDF as annotations
RUN_YASA_NATIVE = True            # extra reference: YASA run directly in this notebook (needs `import yasa`), on the ORIGINAL signal

# --- Manual scoring (optional). Explicit mapping wins: {"<raw edf file name>": "<path to scored edf>"}
MANUAL_SCORING    = {}
MIN_COMMON_PREFIX = 45            # fallback if there is no exact xx.edf <-> xx_scored.edf match: scored EDF whose name shares >= this many leading characters with the raw EDF

# --- Sleepyland paths / API (normally no change needed inside the container)
API_URL      = "http://manager-api:8989"
OUTPUT_ROOTS = [Path("../output"), Path("/app/output")]   # where sleepyland writes <output_name>/<model>/majority/*.npy
OUT_DIR = Path("sleepyland_results")   # writable folder for all results
OUT_TAG = "_sleepyland"
# ------------------------------------------------------

STAGE_NAMES  = ["Wake", "N1", "N2", "N3", "REM"]        # model output order (argmax index)
STAGE_LABELS = ["W", "N1", "N2", "N3", "R"]             # annotation labels
STAGE_COLORS = ["#58e306", "#2cf7f0", "#1173ef", "#4b4d4d", "#ee0e0e"]

## 2. preprocess data
--> Helper functions: channels, folders, signal preparation

In [12]:
def find_channel(ch_names, candidates):
    """First channel matching any candidate (case-insensitive, trailing '.' ignored), else None."""
    lookup = {c.lower().rstrip("."): c for c in ch_names}
    for cand in candidates:
        hit = lookup.get(cand.lower().rstrip("."))
        if hit is not None:
            return hit
    return None


def base_name(path):
    return re.sub(r"(_scored)+$", "", Path(path).stem)


def list_edfs(folder):
    """All *.edf files directly in folder (case-insensitive, sorted, not recursive)."""
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".edf")


def rename_manual_files():
    """Rename every manual scoring in MANUAL_DIR from xx.edf to xx_scored.edf (files already ending in _scored are left alone)."""
    if not MANUAL_DIR.exists():
        print(f"(manual folder {MANUAL_DIR} does not exist - no manual scorings)")
        return
    if MANUAL_DIR.resolve() == RAW_DIR.resolve():
        raise RuntimeError("MANUAL_DIR and RAW_DIR are the same folder: renaming would rename your recordings. "
                           "Please use two different folders.")
    n = 0
    for f in list_edfs(MANUAL_DIR):
        if f.stem.endswith("_scored"):
            continue
        target = f.with_name(f"{f.stem}_scored{f.suffix}")
        if target.exists():
            print(f"  ⚠️ not renamed: {target.name} already exists (kept {f.name})")
            continue
        f.rename(target)
        n += 1
        print(f"  ✏️ renamed manual scoring {f.name} -> {target.name}")
    print(f"manual scorings: {len(list_edfs(MANUAL_DIR))} file(s) in {MANUAL_DIR} ({n} renamed)")


def is_scored_or_manual_file(path):
    """Never treat '*_scored.edf' (manual scoring / earlier output) as a recording."""
    return str(path).endswith("_scored.edf")


def already_scored(raw_path):
    scored = OUT_DIR / f"{base_name(raw_path)}{OUT_TAG}_scored.edf"
    return scored.exists() and scored.stat().st_mtime >= Path(raw_path).stat().st_mtime


def ensure_folders():
    """Originals must NOT stay in API_DIR: predict_one scores every EDF it finds there."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    API_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for f in list_edfs(API_DIR):
        target = RAW_DIR / f.name
        if not AUTO_MOVE_ORIGINALS:
            raise RuntimeError(f"{f} is in {API_DIR}. Move it to {RAW_DIR} (or set AUTO_MOVE_ORIGINALS=True).")
        if target.exists():
            raise RuntimeError(f"{f} is in {API_DIR}, but {target} already exists. Please clean up by hand.")
        shutil.move(str(f), str(target))
        print(f"  ↪ moved {f.name} from {API_DIR} to {RAW_DIR}")


def channel_stats(raw):
    """Amplitude statistics per channel in µV."""
    rows = []
    for ch, x in zip(raw.ch_names, raw.get_data(units="uV")):
        rows.append({"channel": ch, "median_uV": np.median(x), "std_uV": x.std(),
                     "IQR_uV": np.subtract(*np.percentile(x, [75, 25])),
                     "max_abs_uV": np.abs(x).max()})
    return pd.DataFrame(rows).set_index("channel").round(1)


def _runs(mask, sfreq):
    """Lengths (s) of the consecutive True stretches of a boolean mask."""
    d = np.diff(np.r_[0, mask.astype(int), 0])
    return (np.flatnonzero(d == -1) - np.flatnonzero(d == 1)) / sfreq


def _iqr(x):
    return float(np.subtract(*np.percentile(x, [75, 25])))


def prepare_for_sleepyland(raw, eeg, eog):
    """Pick EEG (+EOG), de-saturate, band-pass, resample, clip artefacts, rename to EEG_LABEL / EOG_LABEL.

    Returns (prepared raw, DataFrame with the preprocessing statistics, one row per channel).

    No rescaling: sleepyland applies its own RobustScaler (median 0, IQR 1) and it only works if the
    IQR reflects the EEG and not the DC drift.
    """
    r = raw.copy().pick([eeg] + ([eog] if eog else []))
    rename = {eeg: EEG_LABEL}
    if eog:
        rename[eog] = EOG_LABEL
    r.rename_channels({k: v for k, v in rename.items() if k != v})
    pst = {ch: {} for ch in r.ch_names}           # preprocessing statistics per channel
    sf0 = r.info["sfreq"]

    for ch in r.ch_names:                          # raw amplitude + saturation statistics
        x = r.get_data(picks=ch, units="uV")[0]
        pst[ch].update(raw_median_uV=float(np.median(x)), raw_IQR_uV=_iqr(x))
        sat = np.abs(x) >= SATURATION_UV
        seg = _runs(sat, sf0)
        pst[ch].update(n_sat_samples=int(sat.sum()), pct_saturated=100 * float(sat.mean()),
                       n_sat_segments=int(seg.size), longest_sat_s=float(seg.max()) if seg.size else 0.0,
                       total_sat_s=float(seg.sum()), n_interpolated_samples=0)
        if DESATURATE and sat.any() and not sat.all():   # samples at the ADC limit: linear interpolation
            idx = np.arange(x.size)
            x[sat] = np.interp(idx[sat], idx[~sat], x[~sat])
            r.apply_function(lambda a, x=x: x * 1e-6, picks=[ch])   # data is in volts
            pst[ch]["n_interpolated_samples"] = int(sat.sum())
            print(f"    {ch}: {100 * sat.mean():.3f} % saturated samples interpolated "
                  f"({seg.size} stretch(es), longest {seg.max():.1f} s)")

    if BANDPASS is not None:
        l_freq, h_freq = BANDPASS
        nyq = r.info["sfreq"] / 2
        if h_freq and h_freq >= nyq:
            print(f"    high cut-off {h_freq} Hz >= Nyquist ({nyq:g} Hz): high-pass only")
            h_freq = None
        r.filter(l_freq=l_freq, h_freq=h_freq, picks="all", method="fir", phase="zero", verbose=False)
        print(f"    band-pass {l_freq}-{h_freq if h_freq else 'none'} Hz applied")

    if RESAMPLE_HZ and r.info["sfreq"] != RESAMPLE_HZ:
        r.resample(RESAMPLE_HZ, npad="auto")

    L = int(round(r.info["sfreq"] * EPOCH_SEC))
    for ch in r.ch_names:   # clip artefacts -> small physical range in the exported EDF
        x = r.get_data(picks=ch, units="uV")[0]
        iqr = _iqr(x)
        pst[ch].update(IQR_after_filter_uV=iqr)
        if iqr == 0:
            print(f"    {ch}: IQR = 0 (flat channel?), not clipped")
            continue
        lim = CLIP_IQR_FACTOR * iqr
        clipped = np.abs(x) > lim                  # exactly the samples np.clip changes below
        n_ep = x.size // L
        rms = np.sqrt((np.clip(x, -lim, lim)[:n_ep * L].reshape(n_ep, L) ** 2).mean(1))
        pst[ch].update(clip_limit_uV=lim, pct_samples_clipped=100 * float(clipped.mean()),
                       pct_epochs_with_clipping=100 * float(clipped[:n_ep * L].reshape(n_ep, L).any(1).mean()),
                       pct_epochs_flat=100 * float((rms < 2).mean()),            # epoch RMS < 2 uV
                       pct_epochs_artefact=100 * float((rms > 150).mean()))      # epoch RMS > 150 uV
        print(f"    {ch}: IQR = {iqr:.1f} µV -> clip at ±{lim:.0f} µV ({100 * clipped.mean():.2f} % of samples, "
              f"{pst[ch]['pct_epochs_with_clipping']:.1f} % of epochs contain clipped samples)")
        r.apply_function(lambda a, l=lim * 1e-6: np.clip(a, -l, l), picks=[ch])
    return r, pd.DataFrame(pst).T.rename_axis("channel")


def verify_prepared(path):
    """Read the exported EDF back and warn if it still looks wrong."""
    chk = mne.io.read_raw_edf(str(path), preload=True, verbose=False)
    stats = channel_stats(chk)
    print("  prepared EDF as sleepyland will read it:"); display(stats)
    for ch, row in stats.iterrows():
        if abs(row["median_uV"]) > 5 * max(row["IQR_uV"], 1e-9):
            print(f"  ⚠️ {ch}: median ({row['median_uV']:.0f} µV) >> IQR – DC offset left? Check BANDPASS.")
        if row["IQR_uV"] < 3 or row["IQR_uV"] > 300:
            print(f"  ⚠️ {ch}: IQR = {row['IQR_uV']:.1f} µV is unusual for sleep EEG/EOG.")

## 3. prepare for sleepyland
--> Helper functions: sleepyland API and prediction files

In [13]:
def api_post(endpoint, data=None, params=None):
    """POST to the manager API; raises with the server's message on failure."""
    r = requests.post(f"{API_URL}/{endpoint}", data=data, params=params)
    if r.status_code != 200:
        raise RuntimeError(f"{endpoint} failed ({r.status_code}): {r.text}")
    return r.json()


def run_prediction(prep_edf, output_name, models, channels):
    """Put exactly ONE prepared EDF into API_DIR (the only folder predict_one reads), run predict_one, clean up."""
    leftovers = list(API_DIR.glob("*.edf"))
    if leftovers:
        raise RuntimeError(f"{API_DIR} must be empty before scoring, found: {[f.name for f in leftovers]}")
    for root in OUTPUT_ROOTS:                       # no stale predictions from earlier runs
        shutil.rmtree(root / output_name, ignore_errors=True)
    target = API_DIR / Path(prep_edf).name
    shutil.copy(prep_edf, target)
    try:
        return api_post("predict_one", data={
            "folder_root_name": API_DIR.name,       # ignored by the API (hard-coded /app/input/myedf), kept for the signature
            "folder_name": output_name,
            "channels": list(channels),
            "models": list(models),
            "resolution": str(EPOCH_SEC),
        })
    finally:
        target.unlink(missing_ok=True)


def list_output(output_name):
    """Print every file sleepyland wrote for this output folder (use it if results are not found)."""
    for root in OUTPUT_ROOTS:
        d = root / output_name
        if d.exists():
            print(f"{d}:")
            for p in sorted(d.rglob("*")):
                if p.is_file():
                    print("  ", p.relative_to(d))
            return
    print(f"No output folder '{output_name}' in {[str(r) for r in OUTPUT_ROOTS]}")


def find_prediction(output_name, model):
    """The .npy prediction of a model (majority vote preferred)."""
    for root in OUTPUT_ROOTS:
        base = root / output_name / model
        if base.exists():
            files = sorted((base / "majority").glob("*.npy")) or sorted(base.rglob("*.npy"))
            files = [f for f in files if "TRUE" not in f.name]
            if files:
                return files[0]
    return None


def load_prediction(path):
    """(stages as int array, probabilities [n_epochs x 5] or None)"""
    arr = np.squeeze(np.load(path))
    if arr.ndim == 2:
        return arr.argmax(-1).astype(int), arr
    return arr.astype(int), None


def load_results(output_name, models):
    res = {}
    for m in models:
        p = find_prediction(output_name, m)
        if p is None:
            print(f"    no prediction found for {m}")
        else:
            res[m] = load_prediction(p)
    return res


def run_yasa_native(raw, eeg, eog, emg, n_epochs):
    """YASA directly on the original signal (does its own 0.4-30 Hz filtering)."""
    import yasa
    picks = [c for c in (eeg, eog, emg) if c]
    r = raw.copy().pick(picks)
    r.resample(100, npad="auto")
    sls = yasa.SleepStaging(r, eeg_name=eeg, eog_name=eog, emg_name=emg)
    y = sls.predict()
    if hasattr(y, "as_int"):                                  # yasa >= 0.7: Hypnogram object
        st = np.asarray(y.as_int()).astype(int)
    else:                                                     # yasa <= 0.6: array of 'W','N1',...
        st = pd.Series(np.asarray(y)).map({"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}).to_numpy().astype(int)
    st[(st < 0) | (st > 4)] = 0
    return st[:n_epochs], None

## 4. load manual scorings
Reads stage annotations from the scored EDF and converts them to 30 s epochs. Everything that is not a sleep stage (arousals, …) is ignored. Epochs without a stage get `-1` and are excluded from the metrics.

In [14]:
_STAGE_MAP = {"w": 0, "wake": 0, "1": 1, "n1": 1, "2": 2, "n2": 2, "3": 3, "n3": 3, "4": 3, "n4": 3, "r": 4, "rem": 4}


def _stage_from_desc(desc):
    d = desc.strip().lower().replace("sleep stage", "").replace("stage", "").strip()
    return _STAGE_MAP.get(d)


def manual_hypnogram(scored_edf, n_epochs, epoch_sec=EPOCH_SEC):
    """int array [n_epochs] with 0..4, -1 = not scored. Onsets are relative to the recording start."""
    ann = mne.read_annotations(str(scored_edf))
    h = np.full(n_epochs, -1, dtype=int)
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        s = _stage_from_desc(desc)
        if s is None:
            continue
        a, b = int(round(onset / epoch_sec)), int(round((onset + dur) / epoch_sec))
        h[max(a, 0):min(b, n_epochs)] = s
    return h


def find_manual_edf(raw_path):
    """Manual scoring for a recording: explicit mapping > xx_scored.edf for xx.edf > longest common name prefix."""
    explicit = MANUAL_SCORING.get(Path(raw_path).name)
    if explicit:
        return Path(explicit)
    cands = list_edfs(MANUAL_DIR)
    if not cands:
        return None
    base = base_name(raw_path)
    for p in cands:                                   # exact: xx.edf <-> xx_scored.edf
        if base_name(p) == base:
            return p
    best, best_len = None, 0                          # fallback: longest common prefix of the names
    for p in cands:
        n = len(os.path.commonprefix([base_name(p), base]))
        if n > best_len:
            best, best_len = p, n
    return best if best_len >= MIN_COMMON_PREFIX else None


def evaluate_against_manual(results, manual):
    """Per-model accuracy, kappa, macro-F1, per-stage F1 and confusion matrices vs. the manual hypnogram."""
    from sklearn.metrics import cohen_kappa_score, accuracy_score, f1_score, confusion_matrix
    rows, cms = [], {}
    for m, (st, _) in results.items():
        n = min(len(st), len(manual))
        y, p = manual[:n], np.asarray(st[:n])
        ok = y >= 0
        if ok.sum() == 0:
            continue
        f1 = f1_score(y[ok], p[ok], labels=range(5), average=None, zero_division=0)
        rows.append({"model": m, "n_epochs": int(ok.sum()),
                     "accuracy": accuracy_score(y[ok], p[ok]), "kappa": cohen_kappa_score(y[ok], p[ok]),
                     "macro_F1": f1.mean(), **{f"F1_{s}": v for s, v in zip(STAGE_LABELS, f1)},
                     "%Wake_pred": 100 * (p[ok] == 0).mean()})
        cms[m] = confusion_matrix(y[ok], p[ok], labels=range(5))
    table = pd.DataFrame(rows).set_index("model").round(3) if rows else None
    return table, cms


def confusion_df(cm):
    """Row-normalised (% of manual epochs), rows = manual, columns = prediction."""
    with np.errstate(invalid="ignore", divide="ignore"):
        pct = 100 * cm / cm.sum(1, keepdims=True)
    return pd.DataFrame(np.nan_to_num(pct), index=[f"manual {s}" for s in STAGE_LABELS],
                        columns=[f"pred {s}" for s in STAGE_LABELS]).round(1)

## 5. define YASA_native, statistics (hypnogram & metrics) and plot functions

In [15]:
def _first_run(is_sleep, min_len):
    run = 0
    for i, v in enumerate(is_sleep):
        run = run + 1 if v else 0
        if run >= min_len:
            return i - min_len + 1
    return None


def sleep_statistics(stages, epoch_sec=EPOCH_SEC):
    """YASA's sleep statistics (all durations in minutes), computed from an int hypnogram 0..4.

    Lat_REM is measured from the start of the recording (YASA convention);
    Lat_REM_from_SOL is the AASM definition (from first sleep epoch).
    """
    st = np.asarray(stages)
    epm = 60 / epoch_sec                         # epochs per minute
    is_sleep = st > 0
    idx = np.flatnonzero(is_sleep)
    s = {"TIB": len(st) / epm}
    if idx.size == 0:
        s.update({k: np.nan for k in ["SPT", "WASO", "TST", "SE", "SME", "SFI", "SOL", "SOL_5min"]})
        s["TST"] = 0.0
    else:
        first, last = idx[0], idx[-1]
        s["SPT"] = (last + 1 - first) / epm
        s["WASO"] = np.sum(st[first:last + 1] == 0) / epm
        s["TST"] = is_sleep[first:last + 1].sum() / epm
        s["SE"] = 100 * s["TST"] / s["TIB"]
        s["SME"] = 100 * s["TST"] / s["SPT"]
        s["SFI"] = np.count_nonzero(is_sleep[:-1] & (st[1:] == 0)) / (s["TST"] / 60)
        s["SOL"] = first / epm
        r5 = _first_run(is_sleep, int(np.ceil(5 * epm - 1e-9)))
        s["SOL_5min"] = r5 / epm if r5 is not None else np.nan
    rem = np.flatnonzero(st == 4)
    s["Lat_REM"] = rem[0] / epm if rem.size else np.nan
    s["Lat_REM_from_SOL"] = s["Lat_REM"] - s["SOL"] if rem.size and idx.size else np.nan
    for j, name in enumerate(["WAKE", "N1", "N2", "N3", "REM"]):
        s[name] = np.sum(st == j) / epm
    for name in ["N1", "N2", "N3", "REM"]:
        s[f"%{name}"] = 100 * s[name] / s["TST"] if s["TST"] > 0 else np.nan
    return {k: round(float(v), 4) for k, v in s.items()}


def with_manual(results, manual):
    """results + the manual hypnogram as first entry (unscored epochs counted as Wake for the statistics/plots only)."""
    if manual is None:
        return dict(results)
    return {"manual": (np.where(manual < 0, 0, manual), None), **results}


def summary_table(results):
    return pd.DataFrame({m: sleep_statistics(st) for m, (st, _) in results.items()}).T


def model_agreement(results):
    from sklearn.metrics import cohen_kappa_score
    rows = []
    for a, b in combinations(results, 2):
        sa, sb = results[a][0], results[b][0]
        n = min(len(sa), len(sb))
        rows.append({"pair": f"{a} vs {b}",
                     "agreement_%": 100 * (sa[:n] == sb[:n]).mean(),
                     "kappa": cohen_kappa_score(sa[:n], sb[:n])})
    return pd.DataFrame(rows).set_index("pair").round(2) if rows else None


_YPOS = {0: 4, 4: 3, 1: 2, 2: 1, 3: 0}                  # Wake on top, then REM, N1, N2, N3
_YTICKS = dict(tickvals=[0, 1, 2, 3, 4], ticktext=["N3", "N2", "N1", "REM", "Wake"])


def _ypos(st):
    return [_YPOS.get(int(s), np.nan) for s in st]


def plot_hypnograms(results, title):
    """Interactive plotly hypnograms, one row per model (and the manual scoring, if available)."""
    n = len(results)
    fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=list(results))
    for i, (model, (st, _)) in enumerate(results.items(), start=1):
        t = np.arange(len(st)) * EPOCH_SEC / 3600
        fig.add_trace(go.Scatter(
            x=t, y=_ypos(st), mode="lines+markers",
            line=dict(color="#bdc2c3", width=1.5, shape="hv"),
            marker=dict(size=4, color=[STAGE_COLORS[int(s)] for s in st]),
            customdata=[STAGE_NAMES[int(s)] for s in st],
            hovertemplate="%{x:.2f} h  %{customdata}<extra></extra>",
            showlegend=False), row=i, col=1)
        fig.update_yaxes(range=[-0.5, 4.5], **_YTICKS, row=i, col=1)
    fig.update_xaxes(title_text="Time (h)", row=n, col=1)
    fig.update_layout(title=f"Hypnograms: {title}", height=240 * n + 100)
    fig.show()


def plot_hypnodensity(results, title):
    for model, (_, probs) in results.items():
        if probs is None:
            print(f"{model}: no probabilities available, skipping hypnodensity")
            continue
        t = np.arange(len(probs)) * EPOCH_SEC / 3600
        fig = go.Figure()
        for j, name in enumerate(STAGE_NAMES):
            fig.add_trace(go.Scatter(x=t, y=probs[:, j], name=name, mode="lines", stackgroup="one",
                                     line=dict(width=0.5, color=STAGE_COLORS[j])))
        fig.update_layout(title=f"Hypnodensity: {model} ({title})", xaxis_title="Time (h)",
                          yaxis_title="Probability", yaxis_range=[0, 1], height=350)
        fig.show()


def save_hypnogram_png(results, path, title):
    """Static PNG (like YASA notebook), one panel per model."""
    fig, axes = plt.subplots(len(results), 1, figsize=(12, 2.8 * len(results)),
                             sharex=True, squeeze=False)
    for ax, (model, (st, _)) in zip(axes[:, 0], results.items()):
        ax.step(np.arange(len(st)) * EPOCH_SEC / 3600, _ypos(st), where="post", lw=1)
        ax.set_yticks([0, 1, 2, 3, 4])
        ax.set_yticklabels(["N3", "N2", "N1", "REM", "W"])
        ax.set_title(model, loc="left", fontsize=10)
        ax.grid(alpha=0.3)
    axes[-1, 0].set_xlabel("Time (h)")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)

## 6. define scoring function
`score_edf_sleepyland(edf_path)` returns `{"results": {model: (stages, probs)}, "manual": hypnogram or None, "eval": table or None, "manual_file": name or None, "prep": preprocessing statistics per channel}`.  
Outputs are written to `OUT_DIR`: `<base>_sleepyland_scored.edf` (original, unprocessed data of all channels + hypnogram of `PRIMARY_MODEL` as annotations), `_hypnogram.csv`, `_hypnogram.png`, `_metrics.xlsx` (one sheet per model, plus `vs_manual`), optionally `.set`, and the prepared EDF in `OUT_DIR/prepared`.

In [16]:
def score_edf_sleepyland(edf_path):
    edf_path = str(edf_path)
    if not FORCE_RESCORE and already_scored(edf_path):
        print(f"⏭️  Skipping (already scored & up-to-date): {os.path.basename(edf_path)}")
        return None
    print(f"🔄 Processing: {os.path.basename(edf_path)}")

    # ---- load + channel selection
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    eeg = find_channel(raw.ch_names, EEG_CANDIDATES)
    eog = find_channel(raw.ch_names, EOG_CANDIDATES)
    emg = find_channel(raw.ch_names, EMG_CANDIDATES)
    if eeg is None:
        raise ValueError(f"No EEG channel {EEG_CANDIDATES} found. Available: {raw.ch_names}")
    print(f"→ EEG: {eeg} | EOG: {eog} | EMG: {emg} (EMG only for native YASA)"
          f" | {raw.info['sfreq']:g} Hz | {raw.n_times / raw.info['sfreq'] / 3600:.2f} h")
    if eog is None:
        print("  ⚠️ no EOG channel found, scoring with EEG only")
    n_full = int(raw.n_times / raw.info["sfreq"] // EPOCH_SEC)

    # ---- prepare the sleepyland input
    base = base_name(edf_path)
    safe = re.sub(r"[^A-Za-z0-9]+", "_", base).strip("_")
    output_name = f"out_{safe}"

    print("  before:"); display(channel_stats(raw.copy().pick([c for c in [eeg, eog] if c])))
    prep, prep_stats = prepare_for_sleepyland(raw, eeg, eog)
    prep_stats["duration_h"] = raw.n_times / raw.info["sfreq"] / 3600
    prep_stats["n_epochs"] = n_full
    prep_dir = OUT_DIR / "prepared"
    prep_dir.mkdir(parents=True, exist_ok=True)
    prep_edf = prep_dir / f"{safe}.edf"
    mne.export.export_raw(str(prep_edf), prep, fmt="edf", overwrite=True)
    verify_prepared(prep_edf)

    # ---- score with sleepyland (file is copied into API_DIR only for the duration of the call)
    channels = ["EEG"] + (["EOG"] if eog else [])
    print(f"  scoring with {MODELS} ...")
    run_prediction(prep_edf, output_name, MODELS, channels)
    if not KEEP_PREPARED:
        prep_edf.unlink(missing_ok=True)

    results = load_results(output_name, MODELS)
    if "yasa" not in results and "yasa" in MODELS:
        print("  ℹ️ no YASA result: the container only runs YASA on central EEG labels (C1..C6, Cz). "
              f"Prepared label is '{EEG_LABEL}'. Check: docker compose logs usleepyland | grep -i skipping")
    if RUN_YASA_NATIVE:
        try:
            results["yasa_native"] = run_yasa_native(raw, eeg, eog, emg, n_full)
        except Exception as e:
            print(f"  ⚠️ native YASA failed: {e}")
    if not results:
        list_output(output_name)
        raise RuntimeError("Sleepyland ran, but no prediction files were found (see listing above). "
                           "Look at the container log: docker compose logs usleepyland manager-api")

    for m, (st, _) in results.items():
        if abs(len(st) - n_full) > 1:
            print(f"  ⚠️ {m}: {len(st)} epochs predicted, recording has {n_full} full epochs")

    # ---- manual scoring
    manual, eval_table = None, None
    manual_edf = find_manual_edf(edf_path)
    if manual_edf is not None:
        manual = manual_hypnogram(manual_edf, n_full)
        print(f"  manual scoring: {manual_edf.name} ({(manual >= 0).sum()} of {n_full} epochs scored)")
        eval_table, cms = evaluate_against_manual(results, manual)
    else:
        print("  (no manual scoring found – put the scored EDF into MANUAL_DIR for a quantitative comparison)")

    # ---- outputs (same set as the YASA notebook)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    edf_out = OUT_DIR / f"{base}{OUT_TAG}_scored.edf"
    csv_out = OUT_DIR / f"{base}{OUT_TAG}_hypnogram.csv"
    png_out = OUT_DIR / f"{base}{OUT_TAG}_hypnogram.png"
    xls_out = OUT_DIR / f"{base}{OUT_TAG}_metrics.xlsx"
    set_out = OUT_DIR / f"{base}{OUT_TAG}_scored.set"

    primary = PRIMARY_MODEL if PRIMARY_MODEL in results else next(iter(results))
    if primary != PRIMARY_MODEL:
        print(f"  ⚠️ {PRIMARY_MODEL} has no result, using {primary} for the annotations")
    st_primary = results[primary][0][:n_full]
    raw.set_annotations(mne.Annotations(               # replaces existing annotations, as in the YASA code
        onset=np.arange(len(st_primary)) * EPOCH_SEC,
        duration=[EPOCH_SEC] * len(st_primary),
        description=[STAGE_LABELS[s] for s in st_primary]))
    try:
        raw.export(str(edf_out), fmt="edf", overwrite=True)    # original, unprocessed data of all channels
    except Exception as e:
        print(f"  ⚠️ scored EDF export failed ({e}); the other outputs are still written")

    all_rows = with_manual(results, manual)
    n_max = max(len(st) for st, _ in all_rows.values())
    table = pd.DataFrame({"Time (h)": np.arange(n_max) * EPOCH_SEC / 3600})
    for m, (st, _) in all_rows.items():
        table[m] = pd.Series([STAGE_LABELS[s] for s in st])
    if manual is not None:
        table["manual"] = pd.Series([STAGE_LABELS[s] if s >= 0 else "" for s in manual])
    table.to_csv(csv_out, index=False)

    save_hypnogram_png(all_rows, png_out, f"Hypnogram – {base}")

    stats = {m: sleep_statistics(st) for m, (st, _) in all_rows.items()}
    try:
        with pd.ExcelWriter(xls_out) as xw:
            for m, s in stats.items():
                pd.DataFrame(s, index=[0]).to_excel(xw, sheet_name=m[:31], index=False)
            if eval_table is not None:
                eval_table.to_excel(xw, sheet_name="vs_manual")
    except Exception as e:                                   # e.g. openpyxl missing
        print(f"  ⚠️ Excel export failed ({e}), writing CSV instead")
        pd.DataFrame(stats).T.to_csv(xls_out.with_suffix(".csv"))
        if eval_table is not None:
            eval_table.to_csv(xls_out.with_name(xls_out.stem + "_vs_manual.csv"))

    if EXPORT_SET:
        try:
            mne.export.export_raw(str(set_out), raw, fmt="eeglab", overwrite=True)
            print(f"  💾 Exported EEGLAB .set: {set_out}")
        except Exception as e:
            print(f"  ⚠️ EEGLAB .set export failed: {e}")

    print(f"✅ Done: {edf_out.name}\n")
    return {"results": results, "manual": manual, "eval": eval_table,
            "manual_file": manual_edf.name if manual_edf is not None else None, "prep": prep_stats}

## 7. score all EDFs of the folder with all available models
The files are scored one after the other: sleepyland's `predict_one` always reads one fixed folder (`/app/input/myedf`), so only one prepared file can be in it at a time.

In [17]:
#check if writable
import os, stat, subprocess
print("uid/gid:", os.getuid(), os.getgid())
for p in [API_DIR.parent, API_DIR]:
    s = p.stat()
    print(p.resolve(), oct(stat.S_IMODE(s.st_mode)), "owner uid", s.st_uid,
          "| writable:", os.access(p, os.W_OK))
print(subprocess.run(["ls", "-la", str(API_DIR)], capture_output=True, text=True).stdout)

uid/gid: 1000 100
/app/input 0o777 owner uid 0 | writable: True
/app/input/myedf 0o777 owner uid 1000 | writable: True
total 0
drwxrwxrwx 1 jovyan users 4096 Nov  3 10:33 .
drwxrwxrwx 1 root   root  4096 Nov  3 10:30 ..



In [ ]:
# if myedf is not writable > recreate it (old one is kept as myedf_old). Safe to run again.
if not API_DIR.exists():
    API_DIR.mkdir(parents=True); os.chmod(API_DIR, 0o777)
    print(f"created {API_DIR}")
elif os.access(API_DIR, os.W_OK):
    print(f"{API_DIR} is writable - nothing to do")
else:
    old = API_DIR.with_name("myedf_old")
    if old.exists():
        raise RuntimeError(f"{old} already exists, please clean up by hand (docker compose exec -u root ... chmod -R a+rwX {API_DIR.resolve()})")
    os.rename(API_DIR, old)
    API_DIR.mkdir(); os.chmod(API_DIR, 0o777)
    print(f"{API_DIR} was not writable: recreated it, the old folder is now {old}")

In [ ]:
# score
ensure_folders()
if RENAME_MANUAL:
    rename_manual_files()
try:
    r = requests.get(f"{API_URL}/docs", timeout=5)
    print(f"manager-api reachable (HTTP {r.status_code})")
except Exception as e:
    print(f"⚠️ manager-api not reachable at {API_URL}: {e}")

edf_files = [str(f) for f in list_edfs(RAW_DIR) if not is_scored_or_manual_file(f)]
print(f"Found {len(edf_files)} EDF file(s) in {RAW_DIR.resolve()}")

all_results, failed = {}, {}
for i, f in enumerate(edf_files, 1):
    print(f"\n[{i}/{len(edf_files)}] " + "-" * 60)
    try:
        res = score_edf_sleepyland(f)
        if res:
            all_results[base_name(f)] = res
    except Exception as e:
        failed[f] = e
        print(f"❌ {os.path.basename(f)} failed: {e}\n")
        traceback.print_exc(limit=2)

used = {}                                         # a manual scoring must belong to exactly one recording
for n, o in all_results.items():
    if o["manual_file"]:
        used.setdefault(o["manual_file"], []).append(n)
for mf, names in used.items():
    if len(names) > 1:
        print(f"⚠️ manual scoring {mf} was matched to several recordings: {names} - check names / MANUAL_SCORING")

print(f"\nAll done. Scored: {len(all_results)} | failed: {len(failed)}")

  ✏️ renamed manual scoring 04-49-15-4B-D3-6E_Axo-Device-075_20250707_204037_000009_RvL.edf -> 04-49-15-4B-D3-6E_Axo-Device-075_20250707_204037_000009_RvL_scored.edf
  ✏️ renamed manual scoring 04-49-15-4B-D3-6E_Axo-Device-075_20250708_205434_000010_RvL.edf -> 04-49-15-4B-D3-6E_Axo-Device-075_20250708_205434_000010_RvL_scored.edf
  ✏️ renamed manual scoring 11-E2-7B-2A-A0-71_Axo-Device-071_20250421_204304_000019_RvL.edf -> 11-E2-7B-2A-A0-71_Axo-Device-071_20250421_204304_000019_RvL_scored.edf
  ✏️ renamed manual scoring 11-E2-7B-2A-A0-71_Axo-Device-071_20250504_210352_000029_RvL.edf -> 11-E2-7B-2A-A0-71_Axo-Device-071_20250504_210352_000029_RvL_scored.edf
  ✏️ renamed manual scoring 11-E2-7B-2A-A0-71_Axo-Device-071_20250521_213303_000042_RvL.edf -> 11-E2-7B-2A-A0-71_Axo-Device-071_20250521_213303_000042_RvL_scored.edf
  ✏️ renamed manual scoring 11-E2-7B-2A-A0-71_Axo-Device-071_20250610_210154_000046_RvL.edf -> 11-E2-7B-2A-A0-71_Axo-Device-071_20250610_210154_000046_RvL_scored.edf
  ✏️

,median_uV,std_uV,IQR_uV,max_abs_uV
channel,,,,
FpZ,21672.6,25841.2,46532.4,187500.0
EOG1,8237.0,19723.3,30218.6,187500.0


    C4: 0.432 % saturated samples interpolated (6 stretch(es), longest 108.5 s)
    EOG: 0.066 % saturated samples interpolated (5 stretch(es), longest 10.6 s)
    band-pass 0.3-35.0 Hz applied
    C4: IQR = 23.4 µV -> clip at ±469 µV (1.01 % of samples, 12.0 % of epochs contain clipped samples)
    EOG: IQR = 23.9 µV -> clip at ±478 µV (0.76 % of samples, 13.9 % of epochs contain clipped samples)
  prepared EDF as sleepyland will read it:


,median_uV,std_uV,IQR_uV,max_abs_uV
channel,,,,
C4,-0.5,65.6,23.4,468.5
EOG,-0.2,65.9,23.9,478.5


  scoring with ['usleep', 'deepresnet', 'transformer', 'yasa', 'ensemble'] ...


/opt/conda/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 0.24.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


... (7 outputs left out)

,median_uV,std_uV,IQR_uV,max_abs_uV
channel,,,,
FpZ,-3333.1,6136.9,5046.9,187500.0
EOG1,-4145.7,5924.6,1957.0,187500.0


    C4: 0.062 % saturated samples interpolated (8 stretch(es), longest 11.1 s)
    EOG: 0.077 % saturated samples interpolated (11 stretch(es), longest 15.6 s)
    band-pass 0.3-35.0 Hz applied
    C4: IQR = 25.3 µV -> clip at ±507 µV (0.48 % of samples, 5.6 % of epochs contain clipped samples)
    EOG: IQR = 32.5 µV -> clip at ±649 µV (0.49 % of samples, 5.6 % of epochs contain clipped samples)
  prepared EDF as sleepyland will read it:


,median_uV,std_uV,IQR_uV,max_abs_uV
channel,,,,
C4,0.4,53.8,25.3,506.9
EOG,0.1,69.3,32.5,649.2


  scoring with ['usleep', 'deepresnet', 'transformer', 'yasa', 'ensemble'] ...


## 8. look at results of every single file: plot, stats and compare with manual scoring (optional)
Only runs with `SHOW_PER_FILE = True` (long output for many files). Per file: agreement with the manual scoring, confusion matrices, summary statistics (YASA definitions, minutes), model agreement, interactive hypnograms (manual scoring on top) and hypnodensity graphs. The per-file outputs (EDF with annotations, CSV, PNG, metrics) are saved in `OUT_DIR` by the scoring function in any case; the averages of all these metrics are in section 9.

In [ ]:
if not all_results:
    print("Nothing scored in this run (all files skipped or failed). "
          "Set FORCE_RESCORE = True to re-run files that were already scored.")
elif not SHOW_PER_FILE:
    print(f"Per-file display is switched off (SHOW_PER_FILE = False): {len(all_results)} file(s) scored, see section 9 for the report across files.")
else:
    for name, out in all_results.items():
        res, manual, ev = out["results"], out["manual"], out["eval"]
        print(f"=========== {name} ===========")
        if ev is not None:
            print(f"Agreement with manual scoring (manual: {100 * (manual == 0).sum() / (manual >= 0).sum():.0f} % Wake):")
            display(ev)
            _, cms = evaluate_against_manual(res, manual)
            for m, cm in cms.items():
                print(f"Confusion matrix {m} (% of manual epochs):"); display(confusion_df(cm))
        display(summary_table(with_manual(res, manual)))
        agr = model_agreement(res)
        if agr is not None:
            display(agr)
        plot_hypnograms(with_manual(res, manual), name)
        plot_hypnodensity(res, name)

## 9. report across files
Averages over all scored files (every file counts equally; `±` = standard deviation across files, `n` = number of files that enter the average):  
**9.1** preprocessing statistics (section 2/3 step) · **9.2** agreement with the manual scoring · **9.3** confusion matrices · **9.4** sleep statistics · **9.5** model agreement · **9.6** figure.  
Hypnograms and hypnodensity graphs cannot be averaged and stay per file (`SHOW_PER_FILE = True` or the PNG/CSV files in `OUT_DIR`).
Everything is also written to `OUT_DIR/report_across_files.xlsx` (mean, sd and per-file values).

In [ ]:
import warnings


def _fmt(v):
    """Number with a sensible number of decimals (0.059 / 12.3 / 5945)."""
    if pd.isna(v):
        return "–"
    a = abs(v)
    return f"{v:.{0 if a >= 100 else 1 if a >= 10 else 2 if a >= 1 else 3}f}"


def _mean_sd_str(mean, sd):
    """DataFrame of 'mean ± sd' strings (only the mean if there is a single file)."""
    out = mean.astype(object).copy()
    for r_ in mean.index:
        for c_ in mean.columns:
            m, s = mean.loc[r_, c_], sd.loc[r_, c_]
            out.loc[r_, c_] = _fmt(m) if pd.isna(s) else f"{_fmt(m)} ± {_fmt(s)}"
    return out


def _agg(df, by, cols):
    """mean, sd (ddof=1) and n files per group (groups in order of first appearance)."""
    cols = [c_ for c_ in cols if c_ in df.columns]
    g = df.groupby(by, sort=False)
    return g[cols].mean(), g[cols].std(ddof=1), g["file"].nunique().rename("n")


def collect_report(all_results):
    """Long tables (one row per file x channel/model/pair) of everything section 8 shows."""
    prep, ev, st, agr, cms = [], [], [], [], {}
    for name, out in all_results.items():
        res, manual, e = out["results"], out["manual"], out["eval"]
        if out.get("prep") is not None:
            prep.append(out["prep"].reset_index().assign(file=name))
        st.append(summary_table(with_manual(res, manual)).rename_axis("model").reset_index().assign(file=name))
        a = model_agreement(res)
        if a is not None:
            agr.append(a.reset_index().assign(file=name))
        if e is not None:
            ev.append(e.reset_index().assign(file=name))
            _, cm_dict = evaluate_against_manual(res, manual)
            for m, cm in cm_dict.items():
                with np.errstate(invalid="ignore", divide="ignore"):      # NaN row if a stage is absent in the manual scoring
                    cms.setdefault(m, []).append(100 * cm / cm.sum(1, keepdims=True))
    cat = lambda L: pd.concat(L, ignore_index=True) if L else None
    return {"prep": cat(prep), "eval": cat(ev), "stats": cat(st), "agree": cat(agr), "cms": cms}


PREP_COLS = ["duration_h", "n_epochs", "n_sat_samples", "n_interpolated_samples", "pct_saturated", "n_sat_segments",
             "longest_sat_s", "total_sat_s", "raw_median_uV", "raw_IQR_uV", "IQR_after_filter_uV", "clip_limit_uV",
             "pct_samples_clipped", "pct_epochs_with_clipping", "pct_epochs_flat", "pct_epochs_artefact"]
EVAL_COLS = ["n_epochs", "accuracy", "kappa", "macro_F1"] + [f"F1_{s}" for s in STAGE_LABELS] + ["%Wake_pred"]


def report_across_files(all_results, failed=None):
    if not all_results:
        print("Nothing to report: no file was scored in this run.")
        return None
    rep = collect_report(all_results)
    sheets = {}                                   # name -> DataFrame, saved to Excel at the end

    def add(name, mean, sd, per_file=None):
        sheets[f"{name}_mean"], sheets[f"{name}_sd"] = mean, sd
        if per_file is not None:
            sheets[f"{name}_per_file"] = per_file

    n_manual = sum(o["manual"] is not None for o in all_results.values())
    print(f"Files scored: {len(all_results)} | with manual scoring: {n_manual} | failed: {len(failed or {})}")
    for f_, e_ in (failed or {}).items():
        print(f"   ❌ {os.path.basename(f_)}: {e_}")
    hrs = [o["prep"]["duration_h"].iloc[0] for o in all_results.values() if o.get("prep") is not None]
    if hrs:
        print(f"Recording length: mean {np.mean(hrs):.2f} h (min {np.min(hrs):.2f}, max {np.max(hrs):.2f})")

    # ---- 9.1 preprocessing
    if rep["prep"] is not None:
        mean, sd, n = _agg(rep["prep"], "channel", PREP_COLS)
        print("\n9.1 Preprocessing statistics (mean ± sd across files; per channel of the prepared EDF)")
        tbl = _mean_sd_str(mean, sd).T
        tbl.loc["n files"] = n.astype(int).astype(str)
        display(tbl)
        print("   pct_epochs_with_clipping = share of 30 s epochs that contain at least one clipped sample; "
              "pct_epochs_flat / pct_epochs_artefact = share of epochs with RMS < 2 µV / > 150 µV after clipping.")
        add("prep", mean, sd, rep["prep"].set_index(["file", "channel"]))

    # ---- 9.2 / 9.3 agreement with manual scoring
    if rep["eval"] is not None:
        mean, sd, n = _agg(rep["eval"], "model", EVAL_COLS)
        wake = [100 * (o["manual"] == 0).sum() / (o["manual"] >= 0).sum() for o in all_results.values() if o["manual"] is not None]
        print(f"\n9.2 Agreement with manual scoring (mean ± sd across {n_manual} file(s); manual scoring: {np.mean(wake):.0f} % Wake on average)")
        tbl = _mean_sd_str(mean, sd)
        tbl.insert(0, "n files", n.astype(int))
        display(tbl)
        print("   kappa per file:")
        order = list(mean.index)
        piv = rep["eval"].pivot(index="file", columns="model", values="kappa").reindex(columns=order)
        display(piv.round(3))
        add("vs_manual", mean, sd, rep["eval"].set_index(["file", "model"]))

        print(f"\n9.3 Confusion matrices (% of manual epochs, rows = manual; mean over files of the per-file row-normalised matrices)")
        names_r, names_c = [f"manual {s}" for s in STAGE_LABELS], [f"pred {s}" for s in STAGE_LABELS]
        cm_mean = {}
        for m in order:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN rows (stage never scored manually)
                cm_mean[m] = pd.DataFrame(np.nanmean(np.stack(rep["cms"][m]), axis=0), index=names_r, columns=names_c)
            print(f"   {m}  (n = {len(rep['cms'][m])} files)")
            display(cm_mean[m].round(1))
        sheets["confusion_mean_pct"] = pd.concat(cm_mean)

    # ---- 9.4 sleep statistics
    cols = [c_ for c_ in rep["stats"].columns if c_ not in ("model", "file")]
    mean, sd, n = _agg(rep["stats"], "model", cols)
    print("\n9.4 Sleep statistics (minutes unless %; mean ± sd across files; 'manual' only counts files that have a manual scoring)")
    tbl = _mean_sd_str(mean, sd).T
    tbl.loc["n files"] = n.astype(int).astype(str)
    display(tbl)
    add("sleep_stats", mean, sd, rep["stats"].set_index(["file", "model"]))

    # ---- 9.5 model agreement
    if rep["agree"] is not None:
        mean, sd, n = _agg(rep["agree"], "pair", ["agreement_%", "kappa"])
        print("\n9.5 Agreement between models (mean ± sd across files)")
        tbl = _mean_sd_str(mean, sd)
        tbl.insert(0, "n files", n.astype(int))
        display(tbl)
        add("model_agreement", mean, sd, rep["agree"].set_index(["file", "pair"]))

    # ---- 9.6 figure: kappa per model across files
    if rep["eval"] is not None:
        fig, ax = plt.subplots(figsize=(1.4 * len(order) + 3, 4))
        km, ks = rep["eval"].groupby("model", sort=False)["kappa"].mean(), rep["eval"].groupby("model", sort=False)["kappa"].std(ddof=1)
        ax.bar(range(len(order)), km.reindex(order), yerr=ks.reindex(order).fillna(0), capsize=4, color="#9fb8d9", edgecolor="#4b6a99")
        for i, m in enumerate(order):
            v = rep["eval"].loc[rep["eval"]["model"] == m, "kappa"]
            ax.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, color="k", s=14, zorder=3)
        ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=30, ha="right")
        ax.set_ylabel("Cohen's kappa vs. manual scoring"); ax.set_title(f"Agreement with manual scoring (mean ± sd, dots = files, n = {n_manual})")
        ax.grid(axis="y", alpha=0.3); fig.tight_layout()
        fig.savefig(OUT_DIR / "report_across_files_kappa.png", dpi=200)
        plt.show()

    # ---- save
    xls = OUT_DIR / "report_across_files.xlsx"
    try:
        with pd.ExcelWriter(xls) as xw:
            for name, df in sheets.items():
                df.to_excel(xw, sheet_name=name[:31])
        print(f"\n💾 report saved: {xls}")
    except Exception as e:                        # e.g. openpyxl missing
        folder = OUT_DIR / "report_across_files_csv"
        folder.mkdir(exist_ok=True)
        for name, df in sheets.items():
            df.to_csv(folder / f"{name}.csv")
        print(f"\n⚠️ Excel export failed ({e}); CSV files written to {folder}")
    return rep


report = report_across_files(all_results, failed)